In [ ]:
#to import kaggle envirnoment
import kagglehub
abhishekinnvonix_seizure_epilepcy_chb_mit_eeg_dataset_pediatric_path = kagglehub.dataset_download('abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric')

# IMPORTS

In [ ]:
!pip install mne pywavelets scikit-learn matplotlib > /dev/null
!pip install mne pywavelets scikit-learn pandas > /dev/null

In [ ]:
import os
import mne
import os
import numpy as np
import pandas as pd
import mne
import pywt
import warnings
import gc
from joblib import Parallel, delayed
import shutil

# GLOBALS

In [ ]:
root_dir = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric'
base_path = f'{root_dir}/chb-mit-scalp-eeg-database-1.0.0'

In [ ]:
#Hyperparameters
CONFIG = {
    'BASE_PATH': f'{root_dir}/chb-mit-scalp-eeg-database-1.0.0',  
    'OUTPUT_DIR': '/kaggle/working/processed_features',
    'L_FREQ': 0.5,       # High-pass
    'H_FREQ': 70.0,      # Low-pass
    'EPOCH_DUR': 5.0,    # Window size(s)
    'OVERLAP': 2.5,      # Overlap
    'WAVELET': 'db4',    # Mother Wavelet
    'LEVEL': 5,          # Decomposition Level
    'SAMPLING_RATE': 256,
    'N_JOBS': -1
}

#FOR RECOVERY PART
CONFIG_r = {
    'BASE_PATH': '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric',
    'OUTPUT_DIR': '/kaggle/working/processed_features_fixed', 
    'L_FREQ': 0.5,
    'H_FREQ': 70.0,
    'EPOCH_DUR': 5.0,
    'OVERLAP': 2.5,
    'WAVELET': 'db4',
    'LEVEL': 5,
    'N_JOBS': -1
}

#list of patients that were processed in a wrong way (due to the different input data structure)
TARGET_PATIENTS = ['chb12', 'chb13', 'chb15', 'chb16', 'chb17', 'chb18', 'chb19']

#List of channels that have been selected
STANDARD_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ'
]

# Create output directory
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
print("Setup Completed. Configuration Loaded.")

# DATASET

In [ ]:
#uploading dataset (already uploaded on kaggle servers)
found_path = None
for root, dirs, files in os.walk(root_dir):
    if 'chb01' in dirs:
        found_path = os.path.join(root, 'chb01')
        base_path = root
        print(f"Found in: {base_path}")
        break

if found_path:
    print(f"chb01 data:")
    files = os.listdir(found_path)
    edf_files = [f for f in files if f.endswith('.edf')]
    print(f"Has been found {len(edf_files)} files")
else:
    print("Dataset not uploaded correctly")

## Filtering

In [ ]:
warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

In [ ]:
#As first step we perform filtering (both notch filter and band-pass filter) and then we calculate wdp and wpe
#1. To compute wavelet Packet entropy
def calculate_wpe_batch(data_batch, wavelet='db4', level=5):
    n_epochs, n_channels, n_samples = data_batch.shape
    features = np.zeros((n_epochs, n_channels))

    for ch in range(n_channels):
        for ep in range(n_epochs):
            signal = data_batch[ep, ch, :]
            wp = pywt.WaveletPacket(data=signal, wavelet=wavelet, mode='symmetric', maxlevel=level)
            nodes = [node.path for node in wp.get_level(level, 'freq')]
            energies = np.array([np.sum(np.square(wp[node].data)) for node in nodes])
            total_energy = np.sum(energies)

            if total_energy > 0:
                probabilities = energies / total_energy
                entropy = -np.sum(probabilities * np.log(probabilities + 1e-10))
            else:
                entropy = 0
            features[ep, ch] = entropy
    return features

#To filter/process the input files - Filtering
def process_edf_file(file_path, reference_channels=None):
    try:
        #upload
        raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
        # 1.Remove no EEG files
        eeg_channels = [ch for ch in raw.ch_names if ch not in ['ECG', 'VNS', 'EKG', '-', 'Status']]
        raw.pick_channels(eeg_channels)
        # 2. NOTCH FILTER
        raw.notch_filter(60.0, verbose=False)
        # 3. BAND-PASS filter
        raw.filter(CONFIG['L_FREQ'], CONFIG['H_FREQ'], verbose=False)
        if reference_channels is not None:
            missing_channels = [ch for ch in reference_channels if ch not in raw.ch_names]
            if missing_channels:
                # Se mancano canali fondamentali, saltiamo il file
                return None, None
            raw.pick_channels(reference_channels, ordered=True)
        # 5.creating epochs
        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=CONFIG['EPOCH_DUR'],
            overlap=CONFIG['OVERLAP'],
            verbose=False
        )
        return epochs.get_data(copy=False), raw.ch_names
    except Exception as e:
        print(f"Error in: {os.path.basename(file_path)}: {e}")
        return None, None
#compute wpe for aech channel of a single epoch in parallel
def compute_wpe_single_epoch(epoch_data, wavelet, level):
    n_channels = epoch_data.shape[0]
    epoch_features = np.zeros(n_channels)

    for ch in range(n_channels):
        wp = pywt.WaveletPacket(data=epoch_data[ch, :], wavelet=wavelet, mode='symmetric', maxlevel=level)
        nodes = [node.path for node in wp.get_level(level, 'freq')]

        energies = np.array([np.sum(np.square(wp[node].data)) for node in nodes])
        total_energy = np.sum(energies)

        if total_energy > 0:
            probs = energies / total_energy
            epoch_features[ch] = -np.sum(probs * np.log(probs + 1e-12))
        else:
            epoch_features[ch] = 0
    return epoch_features


#multiprocessing to accelerate run
def calculate_wpe_batch_fast(data_batch, wavelet='db4', level=5):
    results = Parallel(n_jobs=-1)(
        delayed(compute_wpe_single_epoch)(data_batch[i], wavelet, level)
        for i in range(data_batch.shape[0])
    )
    return np.array(results)


def process_problematic_edf(file_path):
    """
    Gestisce i file problematici (chb12, chb13):
    - Cerca i canali standard tra i vari alias (es. T8-P8-1).
    - Carica SOLO quelli.
    - Rinomina tutto allo standard pulito.
    """
    try:
        raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
        current_ch = raw.ch_names
        current_ch_upper = [c.upper().strip() for c in current_ch]
        picks = []
        rename_map = {}
        for std in STANDARD_CHANNELS:
            found = False
            # common alias
            candidates = [std, std.replace('T8-P8', 'T8-P8-1'), std.replace('T8-P8', 'T8-P8-0'),
                          std.replace('P8-O2', 'P8-O2-1'), std.replace('P8-O2', 'P8-O2-0'),
                          std.replace('.', '')]
            for cand in candidates:
                cand_clean = cand.upper()
                if cand_clean in current_ch_upper:
                    idx = current_ch_upper.index(cand_clean)
                    real_name = current_ch[idx]
                    picks.append(real_name)
                    rename_map[real_name] = std
                    found = True
                    break
            if not found:
                return None
        raw.pick_channels(picks, ordered=True)
        raw.load_data(verbose=False)
        raw.rename_channels(rename_map)
        raw.notch_filter(60.0, verbose=False)
        raw.filter(CONFIG_r['L_FREQ'], CONFIG_r['H_FREQ'], verbose=False)
        epochs = mne.make_fixed_length_epochs(
            raw, duration=CONFIG_r['EPOCH_DUR'], overlap=CONFIG_r['OVERLAP'], verbose=False
        )

        return epochs.get_data(copy=False)

    except Exception as e:
        print(f"File error: {os.path.basename(file_path)}: {e}")
        return None

In [ ]:
patient_folders = sorted([f for f in os.listdir(CONFIG['BASE_PATH']) if f.startswith('chb')])

for patient in patient_folders:
    save_path = os.path.join(CONFIG['OUTPUT_DIR'], f"{patient}_wpe_features.npy")
    if os.path.exists(save_path):
        print(f"Skipping {patient}, already done")
        continue

    print(f"\nElaborating patient: {patient}...")
    patient_path = os.path.join(CONFIG['BASE_PATH'], patient)
    edf_files = sorted([f for f in os.listdir(patient_path) if f.endswith('.edf')])

    all_features = []
    patient_ref_channels = None

    for filename in edf_files:
        data, current_channels = process_edf_file(os.path.join(patient_path, filename), patient_ref_channels)

        if data is not None:
            if patient_ref_channels is None:
                patient_ref_channels = current_channels
            # parallel computation
            wpe_batch = Parallel(n_jobs=CONFIG['N_JOBS'])(
                delayed(compute_wpe_single_epoch)(data[i], CONFIG['WAVELET'], CONFIG['LEVEL'])
                for i in range(data.shape[0])
            )
            all_features.append(np.array(wpe_batch))
            del data; gc.collect()

    if all_features:
        full_matrix = np.vstack(all_features)
        np.save(save_path, full_matrix)
        np.save(save_path.replace('.npy', '_channels.npy'), np.array(patient_ref_channels))
        print(f"Saved patient: {patient} w shape: {full_matrix.shape}")
        del all_features; gc.collect()

print("Completed")

Saving in .zip

In [ ]:
#we create a zip of the output folder
shutil.make_archive('CHB_MIT_WPE_Features_complete', 'zip', CONFIG['OUTPUT_DIR'])
print("zip file created, to save it you must download it from the column on the right 'Output'")

# RECUPERO PAZIENTI

In [ ]:
for patient in TARGET_PATIENTS:
    print(f"\Patient: {patient}...")
    patient_path = os.path.join(CONFIG_r['BASE_PATH'], patient)
    if not os.path.exists(patient_path):
        patient_path = os.path.join(CONFIG_r['BASE_PATH'], 'chb-mit-scalp-eeg-database-1.0.0', patient)
    if not os.path.exists(patient_path):
        print(f"Folder not fount for:{patient}")
        continue

    edf_files = sorted([f for f in os.listdir(patient_path) if f.endswith('.edf')])
    all_features = []

    for filename in edf_files:
        full_path = os.path.join(patient_path, filename)
        data = process_problematic_edf(full_path)
        if data is not None:
            # WPE computation
            wpe_batch = Parallel(n_jobs=CONFIG_r['N_JOBS'])(
                delayed(compute_wpe_single_epoch)(data[i], CONFIG_r['WAVELET'], CONFIG_r['LEVEL'])
                for i in range(data.shape[0])
            )
            all_features.append(np.array(wpe_batch))
            del data; gc.collect()
    # saving
    if all_features:
        full_matrix = np.vstack(all_features)
        save_name = os.path.join(CONFIG_r['OUTPUT_DIR'], f"{patient}_wpe_features.npy")
        np.save(save_name, full_matrix)
        np.save(save_name.replace('.npy', '_channels.npy'), np.array(STANDARD_CHANNELS))
        print(f"Saved patient: {patient}, with shape: {full_matrix.shape}")
    else:
        print(f"No data saved for the following patient {patient}")

print("\Patient data recovery completed")

In [ ]:
#we create a zip of the output folder
shutil.make_archive('CHB_MIT_WPE_Features_recovery', 'zip', CONFIG['OUTPUT_DIR'])
print("zip file created, to save it you must download it from the column on the right 'Output'")